In [ ]:
import pandas as pd
import numpy as np
from lstm_dos import lstm_seq2seq
import torch
import matplotlib.pyplot as plt

In [6]:
file_path_1 = "data/vessel_prediction_model_data/gold_model_data/gold_train_data.csv"
file_path_2 = "data/vessel_prediction_model_data/gold_model_data/gold_test_data.csv"
df_train = pd.read_csv(file_path_1)
df_test = pd.read_csv(file_path_2)

In [ ]:
def make_windows_from_df(
    df: pd.DataFrame,
    feature_cols,
    target_cols,
    seq_len: int = 20,
    target_len: int = 5,
    mmsi_col: str = "MMSI",
    time_col: str = "TIME"
):
    """
    Returns:
      X: torch.FloatTensor  (seq_len,  N, n_features)
      Y: torch.FloatTensor  (target_len, N, n_targets)
    """
    df = df.copy()
    df[time_col] = pd.to_datetime(df[time_col])

    X_list = []
    Y_list = []

    for mmsi, g in df.groupby(mmsi_col):
        g = g.sort_values(time_col).reset_index(drop=True)

        feats = g[feature_cols].to_numpy(dtype=np.float32)
        targs = g[target_cols].to_numpy(dtype=np.float32)

        n = len(g)
        if n < seq_len + target_len:
            continue

        for i in range(n - (seq_len + target_len) + 1):
            X_list.append(feats[i : i + seq_len])
            Y_list.append(targs[i + seq_len : i + seq_len + target_len])

    if len(X_list) == 0:
        raise ValueError("No windows created. Increase data or reduce seq_len/target_len.")

    X = torch.from_numpy(np.stack(X_list, axis=0))  # (N, seq_len, n_features)
    Y = torch.from_numpy(np.stack(Y_list, axis=0))  # (N, target_len, 2)

    # Your model expects (seq_len, batch, features)
    X = X.permute(1, 0, 2).contiguous()  # (seq_len, N, n_features)
    Y = Y.permute(1, 0, 2).contiguous()  # (target_len, N, 2)

    return X, Y

In [ ]:
feature_cols = ["LAT","LON","SPEED","dt","cog_sin","cog_cos","hdg_sin","hdg_cos"]
target_cols  = ["LAT","LON"]

seq_len = 20
target_len = 5

X_train, Y_train = make_windows_from_df(df_train, feature_cols, target_cols, seq_len=seq_len, target_len=target_len)
X_test,  Y_test  = make_windows_from_df(df_test,  feature_cols, target_cols, seq_len=seq_len, target_len=target_len)

print("X_train:", X_train.shape, "Y_train:", Y_train.shape)
print("X_test: ", X_test.shape,  "Y_test: ", Y_test.shape)